<h1> Minimal predictive coding network

This notebook shows implementations of a minimial predictive coding network with 1 input, 1 hidden layer and 1 output layer, with 1 neuron per layer. The network is initialized with both weights equal to 1, and is trained for a single iteration on one trainig example (input=1, target=2). This corresponds to network illustrated in Figure 2 of the paper by Song et al (2024, Nature Neuroscience).

The notebook shows a sequence of codes progressing from basic implementation towards the predictive coding library. Each implementation performs exactly the same computations, as evident from the values printed by each code.

Let us start with a simple simulation not using any machine learning or predictive coding library. Indexing of layers starts from 0 for the input layer.

In [ ]:
import numpy as np

# weight initialization
L = 3   #number of layers
w = np.ones(L-1)

# hidden neuron initialization
x1 = 0

# training pattern
x0 = 1
x2 = 2

# hyperparameters
N_relax = 100   # number of relaxation steps
step = 0.1      # size of Euler step during relaxation
lrate = 0.1     # learning rate for the weights

# main code
for i in range(N_relax):
    e1 = x1 - w[0] * x0
    e2 = x2 - w[1] * x1
    x1 = x1 + step * (-e1 + w[1] * e2)

e1 = x1 - w[0] * x0
e2 = x2 - w[1] * x1
w[0] = w[0] + lrate * e1 * x0
w[1] = w[1] + lrate * e2 * x1

print ('Activity of a hidden neuron: ', x1)
print ('Input-to-hidden weight:      ', w[0])
print ('Hidden-to-output weight:     ', w[1])

Activity of a hidden neuron:  1.4999999996944446
Input-to-hidden weight:       1.0499999999694445
Hidden-to-output weight:      1.0750000000305555


Let us now use pytorch auto-differentiation.

In [ ]:
import numpy as np
import torch

# weight initialization
L = 3   #number of layers
w = torch.ones(L-1, requires_grad=True)

# neuronal activity initialization
x1 = torch.tensor(0.0, requires_grad=True)

# training pattern
x0 = 1
x2 = 2

# hyperparameters
N_relax = 100   # number of relaxation steps
step = 0.1      # size of Euler step during relaxation
lrate = 0.1     # learning rate for the weights

# main code
for i in range(N_relax):
    E = 0.5 * ((x2 - w[1]*x1) ** 2 + (x1 - w[0]*x0) ** 2)
    E.backward()
    x1.data = x1.data - step * x1.grad
    x1.grad.zero_() # added, because otherwise the gradients are accumulated

w.grad.zero_() # Reset gradients for w which accumulated above
E = 0.5 * ((x2 - w[1]*x1) ** 2 + (x1 - w[0]*x0) ** 2)
E.backward()
w.data = w.data - lrate * w.grad

print ('Activity of a hidden neuron: ', x1.data)
print ('Input-to-hidden weight:      ', w[0].data)
print ('Hidden-to-output weight:     ', w[1].data)

Activity of a hidden neuron:  tensor(1.5000)
Input-to-hidden weight:       tensor(1.0500)
Hidden-to-output weight:      tensor(1.0750)


Let us now use pythorch optimizers

In [ ]:
import numpy as np
import torch

# weight initialization
L = 3   #number of layers
w = torch.ones(L-1, requires_grad=True)

# neuronal activity initialization
x1 = torch.tensor(0.0, requires_grad=True)

# training pattern
x0 = 1
x2 = 2

# hyperparameters
N_relax = 100   # number of relaxation steps
step = 0.1      # size of Euler step during relaxation
lrate = 0.1     # learning rate for the weights

# main code
optimizer_x = torch.optim.SGD ([x1], lr=step)
optimizer_w = torch.optim.SGD ([w], lr=lrate)
for i in range(N_relax):
    E = 0.5 * ((x2 - w[1]*x1) ** 2 + (x1 - w[0]*x0) ** 2)
    E.backward()
    optimizer_x.step()
    optimizer_x.zero_grad()

optimizer_w.zero_grad() # Reset gradients for w which accumulated above
E = 0.5 * ((x2 - w[1]*x1) ** 2 + (x1 - w[0]*x0) ** 2)
E.backward()
optimizer_w.step()

print ('Activity of a hidden neuron: ', x1.data)
print ('Input-to-hidden weight:      ', w[0].data)
print ('Hidden-to-output weight:     ', w[1].data)

Activity of a hidden neuron:  tensor(1.5000)
Input-to-hidden weight:       tensor(1.0500)
Hidden-to-output weight:      tensor(1.0750)


We now wish to replace our own implementation by a scallable predictive coding (pc) library. Let us start from importing the pc library.

In [ ]:
# import pc library of the code is being run on google colab
try:
  import google.colab
  !git clone https://github.com/Bogacz-Group/PredictiveCoding.git
  ! cp -r PredictiveCoding/predictive_coding predictive_coding
except ImportError:
  pass

Cloning into 'PredictiveCoding'...
remote: Enumerating objects: 235, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 235 (delta 7), reused 6 (delta 1), pack-reused 213 (from 1)
Receiving objects: 100% (235/235), 3.64 MiB | 9.52 MiB/s, done.
Resolving deltas: 100% (111/111), done.


Let us build a neural network using standard object nn.Sequential from pytorch, but containing addiontal pc layer. Furthermore, we replace a training loop by a trainer object from the pc library.

In [ ]:
import numpy as np, torch, predictive_coding as pc
from torch import nn

#defining the network
network = nn.Sequential(
    nn.Linear(1, 1, bias=False),
    pc.PCLayer(),
    nn.Linear(1, 1, bias=False)
)
network.train()   # set the model to training mode

#initializing the weights to the same values as before
network[0].weight.data = torch.ones ((1,1), requires_grad=True)
network[2].weight.data = torch.ones ((1,1), requires_grad=True)

# Defining a loss function
def loss_fn (output, target):
    return 0.5 * (output - target).pow(2).sum()

# hyperparameters
N_relax = 100   # number of relaxation steps
step = 0.1      # size of Euler step during relaxation
lrate = 0.1     # learning rate for the weights

# main code
trainer = pc.PCTrainer(network,
    T = N_relax,
    optimizer_x_fn = torch.optim.SGD,   # optimizer for latent state x
    optimizer_x_kwargs = {'lr': step},  # parameters for latent state optimizer
    optimizer_p_fn = torch.optim.SGD,   # optimizer for parameters
    optimizer_p_kwargs = {'lr': lrate}  # its parameters
)
trainer.train_on_batch(
    inputs = torch.tensor([1.0]),
    loss_fn = loss_fn,
    loss_fn_kwargs = {'target': torch.tensor([2.0])}
)

print ('Activity of a hidden neuron: ', network[1].get_x().data)
print ('Input-to-hidden weight:      ', network[0].weight.data)
print ('Hidden-to-output weight:     ', network[2].weight.data)

Activity of a hidden neuron:  tensor([1.5000])
Input-to-hidden weight:       tensor([[1.0500]])
Hidden-to-output weight:      tensor([[1.0750]])


Although this implementation became slighly longer, it is now fully scalable, as additional layers and elements may be added to the description of the network, analogously as in standard pytorch.